In [1]:
# ── Cell 0: Setup (chạy đầu tiên) ──
import os

# Clone repo nếu chưa có
if not os.path.exists('/content/project'):
    !git clone https://github.com/TruongDuyLongPTIT/CTQW_PRO_METABOLITES_PRIORITIZING.git /content/project

# Add src vào Python path
import sys
sys.path.insert(0, '/content/project/src')

# Install dependencies nếu cần
!pip install -q torch scikit-learn networkx tqdm

Cloning into '/content/project'...
remote: Enumerating objects: 204, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 204 (delta 0), reused 0 (delta 0), pack-reused 203 (from 1)
Receiving objects: 100% (204/204), 1.09 MiB | 11.78 MiB/s, done.
Resolving deltas: 100% (120/120), done.


In [2]:
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print("Drive đã mount sẵn.")

Mounted at /content/drive


In [3]:
!python /content/project/experiments/01_main_results.py

STEP 1 — Build graph
  Graph: 2788 nodes, 22439 edges
  G_pro: 2894 nodes (106 pathway), 31360 edges

STEP 2 — Build eval sets
  hmdb_to_recon: +0 IK, +331 name → 3286 total
  Extracting SMPDB metabolites...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:38<00:00, 1268.19it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  eval_set1 (HMDB+CTD): 158 diseases
  eval_set2 (MarkerDB): 21 diseases
  eval_set3 (SMPDB):    153 diseases

STEP 3 — Eigendecomposition
  Done.

[Table 1] RWR vs CTQW on G_cc...
RWR/HMDB+CTD: 100% 158/158 [12:15<00:00,  4.66s/it]
CTQW/HMDB+CTD: 100% 158/158 [07:56<00:00,  3.01s/it]
  HMDB+CTD: 20.2 min
RWR/MarkerDB: 100% 21/21 [02:12<00:00,  6.32s/it]
CTQW/MarkerDB: 100% 21/21 [01:27<00:00,  4.18s/it]
  MarkerDB: 3.7 min
RWR/SMPDB: 100% 153/153 [09:26<00:00,  3.71s/it]
CTQW/SMPDB: 100% 153/153 [05:53<00:00,  2.31s/it]
  SMPDB: 15.3 min

[Table 2] PROFANCY vs CTQW-PRO on G_pro...
PROFANCY/HMDB+CTD: 100% 158/158 [10:58<00:00,  4.17

In [4]:
!python /content/project/experiments/02_ablation_graph.py

Building graphs...
  G_pro: 2894 → clean: 2851 nodes
Building eval sets...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:50<00:00, 968.88it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
Eigendecomposition...
  eigh(2851×2851)...
  Done in 6.6s

Running ablation...
PROF_o/HMDB+CTD: 100% 158/158 [11:06<00:00,  4.22s/it]
CTQW_o/HMDB+CTD: 100% 158/158 [11:25<00:00,  4.34s/it]
PROF_c/HMDB+CTD: 100% 158/158 [16:30<00:00,  6.27s/it]
CTQW_c/HMDB+CTD: 100% 158/158 [10:46<00:00,  4.09s/it]
  HMDB+CTD: 49.8 min
PROF_o/MarkerDB: 100% 21/21 [02:02<00:00,  5.83s/it]
CTQW_o/MarkerDB: 100% 21/21 [02:03<00:00,  5.88s/it]
PROF_c/MarkerDB: 100% 21/21 [03:00<00:00,  8.57s/it]
CTQW_c/MarkerDB: 100% 21/21 [01:57<00:00,  5.59s/it]
  MarkerDB: 9.1 min
PROF_o/SMPDB: 100% 153/153 [08:15<00:00,  3.24s/it]
CTQW_o/SMPDB: 100% 153/153 [08:33<00:00,  3.36s/it]
PROF_c/SMPDB: 100% 153/153 [12:25<00:00,  4.88s/it]
CTQW_c/SMPDB: 100% 153/153 [08:45<00:00,  3.43s/it]
  SMPDB: 38.0 

In [5]:
!python /content/project/experiments/03_negative_results.py

Setup...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:42<00:00, 1146.24it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158 diseases
  SMPDB:    153 diseases
  Device: cpu

EXP 1: Self-loop leakage analysis (GPU-accelerated)
  (A) Baseline:     no self-loops (γ=0)
  (B) Leaked:       H = A_pro + 10.0·diag(ALL disease mets in G_pro)
  (C) Leakage-free: H = A_pro + 10.0·diag(seed mets only, per fold)
  Semantics of B: test_met receives self-loop → score artificially boosted
  Dataset: full SMPDB (153 diseases)
  NOTE: uses torch.linalg.eigh (symmetric H) for all 3 conditions
  [ 10/153] elapsed=20.3m  ETA=290.2m | MRR  A=0.2561  B=0.3759  C=0.2379 | C_eigh=154.4s
  [ 20/153] elapsed=44.6m  ETA=296.9m | MRR  A=0.2388  B=0.3278  C=0.2275 | C_eigh=79.5s
  [ 30/153] elapsed=63.2m  ETA=259.2m | MRR  A=0.2460  B=0.3280  C=0.2335 | C_eigh=97.6s
  [ 40/153] elapsed=84.4m  ETA=238.6m | MRR  A=0.2318  B=0.3154  C=0.2192 | C_eigh=92.8s
  [ 50/153

In [6]:
!python /content/project/experiments/06_biological_interpretability.py

06 — Biological Interpretability Analysis
Setup...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:43<00:00, 1126.99it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215

Building NH-CTQW-PRO (γ=22.0, t=0.1)...
  Done in 101.2s

DISEASE: Lesch-Nyhan Syndrome (LNS)  [LNS]

Seeds (n=24) — known disease metabolites:
    #  Name                                                         HMDB  HMDB link
  -----------------------------------------------------------------------------------------------
    1. (S)-2-[5-Amino-1-(5-phospho-D-ribosyl)imidazole-4-carboxamido]succinate    HMDB0000797  https://hmdb.ca/metabolites/HMDB0000797
    2. Adenine                                               HMDB0000034  https://hmdb.ca/metabolites/HMDB0000034
    3. Adenosine                                             HMDB0004402  https://hmdb.ca/metabolites/HMDB0004402
    4. 5-Amino-1-(5-Phospho-D-ribosyl)imidazole-4-carboxamide    HMDB0001517  https://hmdb.ca/metabolites/H